# Feature Engineering

This notebook transforms the NFLverse regular-season schedule data into a dataset suitable for machine learning.

The main goal is to create **pre-game features**, meaning that every feature used to predict a game is based only on information that was available before that game was played.

## Creating Team-Level Game Data

The original schedule contains one row per game. To calculate team performance statistics, each game is represented from both the home team's and away team's perspective.

For each team-game observation, we record:

- Points scored
- Points allowed
- Whether the team won
- Season and week
- Opposing team

This allows us to calculate each team's performance throughout a season.

## Calculating Pre-Game Statistics

Cumulative team statistics are calculated for each season and team, including:

- Games played
- Wins
- Win percentage
- Points per game (PPG)
- Points allowed per game (PAPG)

The statistics are shifted by one game so that the current game is **not included** in its own features.

For example, the features for Week 8 are based only on the team's performance from Weeks 1–7.

This prevents **data leakage** and better represents how the model would operate in a real prediction scenario.

## Creating Matchup Features

The pre-game statistics for the home and away teams are joined back to the original game data.

Three matchup features are created:

- `win_pct_diff` — home team's win percentage minus away team's win percentage
- `ppg_diff` — home team's points per game minus away team's points per game
- `defense_diff` — away team's points allowed per game minus home team's points allowed per game

Positive values generally indicate an advantage for the home team.

## Target Variable

A binary `home_win` target is created from the `result` column.

- `result > 0` → home team wins → `home_win = 1`
- `result < 0` → away team wins → `home_win = 0`

This will be the target variable that the machine learning models attempt to predict.

## Handling Missing Values

Week 1 games do not have previous games within the same season, so pre-game team statistics are unavailable.

For this initial model, games with missing values in the engineered features are removed.

A later version of the model can improve this by incorporating information from the previous season, allowing predictions to be made for Week 1.

## Chronological Data Split

The data is split chronologically to simulate a realistic prediction environment:

- **2015–2022:** Training set
- **2023:** Validation set
- **2024–2025:** Test set

A chronological split is used instead of a random split because future games should never influence predictions for earlier games.

The final features are converted from Polars to NumPy only at this stage so they can be used with scikit-learn.

## Result

The notebook produces a leakage-free dataset containing pre-game team performance and matchup statistics.

The initial feature set consists of:

- `win_pct_diff`
- `ppg_diff`
- `defense_diff`

These features, together with the `home_win` target, are ready to be used for the first machine learning models in the next notebook.

## 1. Imports

In [151]:
import nflreadpy as nfl
import polars as pl

## 2. Load NFL Schedule Data

We use regular-season games from 2015 through 2025.

The schedule contains the game outcome, teams, scores, dates, rest information, quarterback information, and other game-level variables.

In [152]:
schedules = nfl.load_schedules(
    seasons=range(2015, 2026)
)

games = schedules.filter(
    pl.col("game_type") == "REG"
)

print(games.shape)

(2895, 46)


In [153]:
games.columns

['game_id',
 'season',
 'game_type',
 'week',
 'gameday',
 'weekday',
 'gametime',
 'away_team',
 'away_score',
 'home_team',
 'home_score',
 'location',
 'result',
 'total',
 'overtime',
 'old_game_id',
 'gsis',
 'nfl_detail_id',
 'pfr',
 'pff',
 'espn',
 'ftn',
 'away_rest',
 'home_rest',
 'away_moneyline',
 'home_moneyline',
 'spread_line',
 'away_spread_odds',
 'home_spread_odds',
 'total_line',
 'under_odds',
 'over_odds',
 'div_game',
 'roof',
 'surface',
 'temp',
 'wind',
 'away_qb_id',
 'home_qb_id',
 'away_qb_name',
 'home_qb_name',
 'away_coach',
 'home_coach',
 'referee',
 'stadium_id',
 'stadium']

In [154]:
games.schema

Schema([('game_id', String),
        ('season', Int32),
        ('game_type', String),
        ('week', Int32),
        ('gameday', String),
        ('weekday', String),
        ('gametime', String),
        ('away_team', String),
        ('away_score', Int32),
        ('home_team', String),
        ('home_score', Int32),
        ('location', String),
        ('result', Int32),
        ('total', Int32),
        ('overtime', Int32),
        ('old_game_id', String),
        ('gsis', Int32),
        ('nfl_detail_id', String),
        ('pfr', String),
        ('pff', Int32),
        ('espn', String),
        ('ftn', Int32),
        ('away_rest', Int32),
        ('home_rest', Int32),
        ('away_moneyline', Int32),
        ('home_moneyline', Int32),
        ('spread_line', Float64),
        ('away_spread_odds', Int32),
        ('home_spread_odds', Int32),
        ('total_line', Float64),
        ('under_odds', Int32),
        ('over_odds', Int32),
        ('div_game', Int32),
        ('r

## 3. Create Team-Level Game Data

Each game is converted into two observations:

- one from the home team's perspective
- one from the away team's perspective

This allows us to calculate statistics for each team over time.

In [155]:
home_games = games.select([
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    pl.col("home_team").alias("team"),
    pl.col("home_score").alias("points_for"),
    pl.col("away_score").alias("points_against"),
    (pl.col("result") > 0).cast(pl.Int8).alias("win"),
])

home_games.head()

game_id,season,week,home_team,away_team,team,points_for,points_against,win
str,i32,i32,str,str,str,i32,i32,i8
"""2015_01_PIT_NE""",2015,1,"""NE""","""PIT""","""NE""",28,21,1
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""","""BUF""",27,14,1
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""","""CHI""",23,31,0
"""2015_01_KC_HOU""",2015,1,"""HOU""","""KC""","""HOU""",20,27,0
"""2015_01_CAR_JAX""",2015,1,"""JAX""","""CAR""","""JAX""",9,20,0


In [156]:
away_games = games.select([
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    pl.col("away_team").alias("team"),
    pl.col("away_score").alias("points_for"),
    pl.col("home_score").alias("points_against"),
    (pl.col("result") < 0).cast(pl.Int8).alias("win"),
])

away_games.head()

game_id,season,week,home_team,away_team,team,points_for,points_against,win
str,i32,i32,str,str,str,i32,i32,i8
"""2015_01_PIT_NE""",2015,1,"""NE""","""PIT""","""PIT""",21,28,0
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""","""IND""",14,27,0
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""","""GB""",31,23,1
"""2015_01_KC_HOU""",2015,1,"""HOU""","""KC""","""KC""",27,20,1
"""2015_01_CAR_JAX""",2015,1,"""JAX""","""CAR""","""CAR""",20,9,1


In [157]:
team_games = pl.concat([
    home_games,
    away_games
]).sort([
    "season",
    "team",
    "week"
])

team_games.head(10)

game_id,season,week,home_team,away_team,team,points_for,points_against,win
str,i32,i32,str,str,str,i32,i32,i8
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""","""ARI""",31,19,1
"""2015_02_ARI_CHI""",2015,2,"""CHI""","""ARI""","""ARI""",48,23,1
"""2015_03_SF_ARI""",2015,3,"""ARI""","""SF""","""ARI""",47,7,1
"""2015_04_STL_ARI""",2015,4,"""ARI""","""STL""","""ARI""",22,24,0
"""2015_05_ARI_DET""",2015,5,"""DET""","""ARI""","""ARI""",42,17,1
"""2015_06_ARI_PIT""",2015,6,"""PIT""","""ARI""","""ARI""",13,25,0
"""2015_07_BAL_ARI""",2015,7,"""ARI""","""BAL""","""ARI""",26,18,1
"""2015_08_ARI_CLE""",2015,8,"""CLE""","""ARI""","""ARI""",34,20,1
"""2015_10_ARI_SEA""",2015,10,"""SEA""","""ARI""","""ARI""",39,32,1


In [158]:
team_games.shape

(5790, 9)

## 4. Season-to-Date Team Statistics

For every game, we calculate the team's performance before that game.

Features:

- Wins before the game
- Games played before the game
- Points scored before the game
- Points allowed before the game
- Win percentage
- Points per game
- Points allowed per game

The current game is excluded to prevent data leakage.

In [159]:
team_games = team_games.with_columns([
    pl.col("win")
    .cum_sum()
    .over(["season", "team"])
    .shift(1)
    .over(["season", "team"])
    .alias("wins_before"),

    pl.col("points_for")
    .cum_sum()
    .over(["season", "team"])
    .shift(1)
    .over(["season", "team"])
    .alias("points_for_before"),

    pl.col("points_against")
    .cum_sum()
    .over(["season", "team"])
    .shift(1)
    .over(["season", "team"])
    .alias("points_against_before"),

    pl.col("win")
    .cum_count()
    .over(["season", "team"])
    .shift(1)
    .over(["season", "team"])
    .alias("games_before"),
])

In [160]:
team_games.head(10)

game_id,season,week,home_team,away_team,team,points_for,points_against,win,wins_before,points_for_before,points_against_before,games_before
str,i32,i32,str,str,str,i32,i32,i8,i64,i32,i32,u32
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""","""ARI""",31,19,1,null,null,null,null
"""2015_02_ARI_CHI""",2015,2,"""CHI""","""ARI""","""ARI""",48,23,1,1,31,19,1
"""2015_03_SF_ARI""",2015,3,"""ARI""","""SF""","""ARI""",47,7,1,2,79,42,2
"""2015_04_STL_ARI""",2015,4,"""ARI""","""STL""","""ARI""",22,24,0,3,126,49,3
"""2015_05_ARI_DET""",2015,5,"""DET""","""ARI""","""ARI""",42,17,1,3,148,73,4
"""2015_06_ARI_PIT""",2015,6,"""PIT""","""ARI""","""ARI""",13,25,0,4,190,90,5
"""2015_07_BAL_ARI""",2015,7,"""ARI""","""BAL""","""ARI""",26,18,1,4,203,115,6
"""2015_08_ARI_CLE""",2015,8,"""CLE""","""ARI""","""ARI""",34,20,1,5,229,133,7
"""2015_10_ARI_SEA""",2015,10,"""SEA""","""ARI""","""ARI""",39,32,1,6,263,153,8


In [161]:
team_games = team_games.with_columns([
    (
        pl.col("wins_before") /
        pl.col("games_before")
    ).alias("win_pct_before"),

    (
        pl.col("points_for_before") /
        pl.col("games_before")
    ).alias("ppg_before"),

    (
        pl.col("points_against_before") /
        pl.col("games_before")
    ).alias("papg_before"),
])

## 5. Rolling Five-Game Performance

Season-long statistics can hide recent changes in team performance.

We therefore calculate statistics based on the team's previous five games:

- Rolling win percentage
- Rolling points per game
- Rolling points allowed per game
- Rolling point differential per game

The current game is excluded from every calculation.

In [162]:
team_games = team_games.with_columns([
    pl.col("win")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_win_pct_5"),

    pl.col("points_for")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_ppg_5"),

    pl.col("points_against")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_papg_5"),

    (
        (pl.col("points_for") - pl.col("points_against"))
        .shift(1)
        .rolling_mean(window_size=5, min_samples=1)
        .over(["season", "team"])
    ).alias("rolling_point_diff_5"),
])

In [163]:
team_games.select([
    "season",
    "week",
    "team",
    "win",
    "rolling_win_pct_5",
    "rolling_ppg_5",
    "rolling_papg_5",
    "rolling_point_diff_5",
]).head(15)

season,week,team,win,rolling_win_pct_5,rolling_ppg_5,rolling_papg_5,rolling_point_diff_5
i32,i32,str,i8,f64,f64,f64,f64
2015,1,"""ARI""",1,null,null,null,null
2015,2,"""ARI""",1,1.0,31.0,19.0,12.0
2015,3,"""ARI""",1,1.0,39.5,21.0,18.5
2015,4,"""ARI""",0,1.0,42.0,16.333333,25.666667
2015,5,"""ARI""",1,0.75,37.0,18.25,18.75
…,…,…,…,…,…,…,…
2015,12,"""ARI""",1,0.8,29.2,25.2,4.0
2015,13,"""ARI""",1,1.0,30.4,22.8,7.6
2015,14,"""ARI""",1,1.0,30.6,19.8,10.8


## 6. Create Team Feature Table

We now select the team-level statistics that will later be joined to each game from the home and away perspectives.

In [164]:
team_features = team_games.select([
    "game_id",
    "team",

    "win_pct_before",
    "ppg_before",
    "papg_before",

    "rolling_win_pct_5",
    "rolling_ppg_5",
    "rolling_papg_5",
    "rolling_point_diff_5",
])

In [165]:
home_features = team_features.rename({
    "team": "home_team",

    "win_pct_before": "home_win_pct",
    "ppg_before": "home_ppg",
    "papg_before": "home_papg",

    "rolling_win_pct_5": "home_rolling_win_pct_5",
    "rolling_ppg_5": "home_rolling_ppg_5",
    "rolling_papg_5": "home_rolling_papg_5",
    "rolling_point_diff_5": "home_rolling_point_diff_5",
})

In [166]:
away_features = team_features.rename({
    "team": "away_team",

    "win_pct_before": "away_win_pct",
    "ppg_before": "away_ppg",
    "papg_before": "away_papg",

    "rolling_win_pct_5": "away_rolling_win_pct_5",
    "rolling_ppg_5": "away_rolling_ppg_5",
    "rolling_papg_5": "away_rolling_papg_5",
    "rolling_point_diff_5": "away_rolling_point_diff_5",
})

## 7. Combine Home and Away Features

The home and away team statistics are joined to the original game-level dataset.

The target variable is:

- `home_win = 1` if the home team wins
- `home_win = 0` otherwise

In [167]:
model_data = (
    games
    .with_columns(
        (pl.col("result") > 0)
        .cast(pl.Int8)
        .alias("home_win")
    )
    .join(
        home_features,
        on=["game_id", "home_team"],
        how="left"
    )
    .join(
        away_features,
        on=["game_id", "away_team"],
        how="left"
    )
)

## 8. Create Matchup Features

Rather than giving the model separate home and away statistics, we calculate the difference between the teams.

Positive values generally indicate an advantage for the home team.

For defensive statistics, a lower points-allowed value is better, so the difference is calculated as:

away PAPG - home PAPG

In [168]:
model_data = model_data.with_columns([
    (
        pl.col("home_win_pct") -
        pl.col("away_win_pct")
    ).alias("win_pct_diff"),

    (
        pl.col("home_ppg") -
        pl.col("away_ppg")
    ).alias("ppg_diff"),

    (
        pl.col("away_papg") -
        pl.col("home_papg")
    ).alias("defense_diff"),

    (
        pl.col("home_rolling_win_pct_5") -
        pl.col("away_rolling_win_pct_5")
    ).alias("rolling_win_pct_diff_5"),

    (
        pl.col("home_rolling_ppg_5") -
        pl.col("away_rolling_ppg_5")
    ).alias("rolling_ppg_diff_5"),

    (
        pl.col("away_rolling_papg_5") -
        pl.col("home_rolling_papg_5")
    ).alias("rolling_defense_diff_5"),

    (
        pl.col("home_rolling_point_diff_5") -
        pl.col("away_rolling_point_diff_5")
    ).alias("rolling_point_diff_diff_5"),
])

## 9. V3 — Play-by-Play Efficiency

Points scored and points allowed do not fully describe how efficiently a team performs.

Expected Points Added (EPA) measures the change in expected points resulting from a play.

We will use NFL play-by-play data to create pre-game offensive and defensive efficiency features.

Only plays from games occurring before the game being predicted will be used.

In [169]:
pbp = nfl.load_pbp(
    seasons=range(2015, 2026)
)

print(pbp.shape)

(532376, 372)


In [170]:
print(pbp.columns)

['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln', 'ydstogo', 'ydsnet', 'desc', 'play_type', 'yards_gained', 'shotgun', 'no_huddle', 'qb_dropback', 'qb_kneel', 'qb_spike', 'qb_scramble', 'pass_length', 'pass_location', 'air_yards', 'yards_after_catch', 'run_location', 'run_gap', 'field_goal_result', 'kick_distance', 'extra_point_result', 'two_point_conv_result', 'home_timeouts_remaining', 'away_timeouts_remaining', 'timeout', 'timeout_team', 'td_team', 'td_player_name', 'td_player_id', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining', 'total_home_score', 'total_away_score', 'posteam_score', 'defteam_score', 'score_differential', 'posteam_score_post', 'defteam_score_post', 'score_differential

## 10. Calculate Game-Level EPA

For each team in each game, calculate:

- Offensive EPA per play
- Defensive EPA allowed per play
- Offensive EPA per pass play
- Offensive EPA per rush play

EPA is calculated from individual plays and then aggregated to the game level.

Only regular-season plays are included.

In [171]:
pbp_regular = pbp.filter(
    pl.col("season_type") == "REG"
)

In [172]:
offensive_plays = pbp_regular.filter(
    pl.col("posteam").is_not_null()
)

In [173]:
offense_game_epa = (
    offensive_plays
    .group_by([
        "game_id",
        "season",
        "week",
        "posteam",
    ])
    .agg([
        pl.col("epa").sum().alias("off_epa"),
        pl.col("epa").mean().alias("off_epa_per_play"),
        pl.len().alias("off_plays"),

        pl.when(pl.col("pass_attempt") == 1)
        .then(pl.col("epa"))
        .otherwise(None)
        .mean()
        .alias("pass_epa_per_play"),

        pl.when(pl.col("rush_attempt") == 1)
        .then(pl.col("epa"))
        .otherwise(None)
        .mean()
        .alias("rush_epa_per_play"),
    ])
    .rename({
        "posteam": "team"
    })
)

In [174]:
defense_game_epa = (
    offensive_plays
    .group_by([
        "game_id",
        "season",
        "week",
        "defteam",
    ])
    .agg([
        pl.col("epa").sum().alias("def_epa_allowed"),
        pl.col("epa").mean().alias("def_epa_allowed_per_play"),
        pl.len().alias("def_plays"),
    ])
    .rename({
        "defteam": "team"
    })
)

In [175]:
team_game_epa = (
    offense_game_epa
    .join(
        defense_game_epa,
        on=["game_id", "season", "week", "team"],
        how="inner"
    )
)

In [176]:
team_game_epa = team_game_epa.sort([
    "season",
    "team",
    "week"
])

team_game_epa.head()

game_id,season,week,team,off_epa,off_epa_per_play,off_plays,pass_epa_per_play,rush_epa_per_play,def_epa_allowed,def_epa_allowed_per_play,def_plays
str,i32,i32,str,f64,f64,u32,f64,f64,f64,f64,u32
"""2015_01_NO_ARI""",2015,1,"""ARI""",11.272465,0.144519,79,0.507832,-0.348388,1.140083,0.012392,92
"""2015_02_ARI_CHI""",2015,2,"""ARI""",23.54183,0.313891,76,0.46972,0.060615,-6.891672,-0.074909,92
"""2015_03_SF_ARI""",2015,3,"""ARI""",12.296117,0.135122,92,0.341914,0.134756,-29.025881,-0.433222,67
"""2015_04_STL_ARI""",2015,4,"""ARI""",-2.790067,-0.031705,88,-0.013202,0.007977,0.54902,0.007419,75
"""2015_05_ARI_DET""",2015,5,"""ARI""",11.212028,0.169879,67,0.331412,0.330461,-19.752516,-0.176362,112


In [177]:
team_game_epa = team_game_epa.with_columns([
    # Season-to-date offensive EPA/play
    (
        pl.col("off_epa_per_play")
        .cum_sum()
        .over(["season", "team"])
        .shift(1)
        .over(["season", "team"])
        /
        pl.col("off_epa_per_play")
        .cum_count()
        .over(["season", "team"])
        .shift(1)
        .over(["season", "team"])
    ).alias("off_epa_per_play_before"),

    # Season-to-date defensive EPA/play
    (
        pl.col("def_epa_allowed_per_play")
        .cum_sum()
        .over(["season", "team"])
        .shift(1)
        .over(["season", "team"])
        /
        pl.col("def_epa_allowed_per_play")
        .cum_count()
        .over(["season", "team"])
        .shift(1)
        .over(["season", "team"])
    ).alias("def_epa_allowed_per_play_before"),

    # Rolling 5 games
    pl.col("off_epa_per_play")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_off_epa_per_play_5"),

    pl.col("def_epa_allowed_per_play")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_def_epa_allowed_per_play_5"),

    pl.col("pass_epa_per_play")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_pass_epa_per_play_5"),

    pl.col("rush_epa_per_play")
    .shift(1)
    .rolling_mean(window_size=5, min_samples=1)
    .over(["season", "team"])
    .alias("rolling_rush_epa_per_play_5"),
])

In [178]:
team_game_epa.filter(
    pl.col("team") == "KC"
).select([
    "season",
    "week",
    "team",
    "off_epa_per_play",
    "off_epa_per_play_before",
    "rolling_off_epa_per_play_5",
    "def_epa_allowed_per_play",
    "def_epa_allowed_per_play_before",
    "rolling_def_epa_allowed_per_play_5",
]).head(10)

season,week,team,off_epa_per_play,off_epa_per_play_before,rolling_off_epa_per_play_5,def_epa_allowed_per_play,def_epa_allowed_per_play_before,rolling_def_epa_allowed_per_play_5
i32,i32,str,f64,f64,f64,f64,f64,f64
2015,1,"""KC""",-0.049293,null,null,-0.165757,null,null
2015,2,"""KC""",-0.111948,-0.049293,-0.049293,-0.013721,-0.165757,-0.165757
2015,3,"""KC""",0.074551,-0.08062,-0.08062,0.169254,-0.089739,-0.089739
2015,4,"""KC""",0.104391,-0.028897,-0.028897,0.290004,-0.003408,-0.003408
2015,5,"""KC""",-0.073994,0.004425,0.004425,-0.023722,0.069945,0.069945
2015,6,"""KC""",-0.118692,-0.011259,-0.011259,-0.0427,0.051212,0.051212
2015,7,"""KC""",0.073153,-0.029164,-0.025139,-0.047189,0.03556,0.075823
2015,8,"""KC""",0.33609,-0.014548,0.011882,-0.1341,0.023738,0.069129
2015,10,"""KC""",0.019075,0.029282,0.064189,-0.217751,0.004009,0.008459


In [179]:
team_epa_features = team_game_epa.select([
    "game_id",
    "team",

    "off_epa_per_play_before",
    "def_epa_allowed_per_play_before",

    "rolling_off_epa_per_play_5",
    "rolling_def_epa_allowed_per_play_5",
    "rolling_pass_epa_per_play_5",
    "rolling_rush_epa_per_play_5",
])

In [180]:
home_epa_features = team_epa_features.rename({
    "team": "home_team",

    "off_epa_per_play_before": "home_off_epa",
    "def_epa_allowed_per_play_before": "home_def_epa",

    "rolling_off_epa_per_play_5": "home_rolling_off_epa_5",
    "rolling_def_epa_allowed_per_play_5": "home_rolling_def_epa_5",
    "rolling_pass_epa_per_play_5": "home_rolling_pass_epa_5",
    "rolling_rush_epa_per_play_5": "home_rolling_rush_epa_5",
})

away_epa_features = team_epa_features.rename({
    "team": "away_team",

    "off_epa_per_play_before": "away_off_epa",
    "def_epa_allowed_per_play_before": "away_def_epa",

    "rolling_off_epa_per_play_5": "away_rolling_off_epa_5",
    "rolling_def_epa_allowed_per_play_5": "away_rolling_def_epa_5",
    "rolling_pass_epa_per_play_5": "away_rolling_pass_epa_5",
    "rolling_rush_epa_per_play_5": "away_rolling_rush_epa_5",
})

In [181]:
model_data = (
    model_data
    .join(
        home_epa_features,
        on=["game_id", "home_team"],
        how="left"
    )
    .join(
        away_epa_features,
        on=["game_id", "away_team"],
        how="left"
    )
)

In [182]:
model_data.select([
    "game_id",
    "home_team",
    "away_team",
    "home_off_epa",
    "away_off_epa",
    "home_def_epa",
    "away_def_epa",
]).head(10)

game_id,home_team,away_team,home_off_epa,away_off_epa,home_def_epa,away_def_epa
str,str,str,f64,f64,f64,f64
"""2015_01_PIT_NE""","""NE""","""PIT""",null,null,null,null
"""2015_01_IND_BUF""","""BUF""","""IND""",null,null,null,null
"""2015_01_GB_CHI""","""CHI""","""GB""",null,null,null,null
"""2015_01_KC_HOU""","""HOU""","""KC""",null,null,null,null
"""2015_01_CAR_JAX""","""JAX""","""CAR""",null,null,null,null
"""2015_01_CLE_NYJ""","""NYJ""","""CLE""",null,null,null,null
"""2015_01_SEA_STL""","""STL""","""SEA""",null,null,null,null
"""2015_01_MIA_WAS""","""WAS""","""MIA""",null,null,null,null
"""2015_01_NO_ARI""","""ARI""","""NO""",null,null,null,null


In [183]:
model_data = model_data.with_columns([
    # Offensive EPA
    (
        pl.col("home_off_epa") -
        pl.col("away_off_epa")
    ).alias("off_epa_diff"),

    # Defensive EPA
    (
        pl.col("away_def_epa") -
        pl.col("home_def_epa")
    ).alias("def_epa_diff"),

    # Rolling offensive EPA
    (
        pl.col("home_rolling_off_epa_5") -
        pl.col("away_rolling_off_epa_5")
    ).alias("rolling_off_epa_diff_5"),

    # Rolling defensive EPA
    (
        pl.col("away_rolling_def_epa_5") -
        pl.col("home_rolling_def_epa_5")
    ).alias("rolling_def_epa_diff_5"),

    # Rolling passing EPA
    (
        pl.col("home_rolling_pass_epa_5") -
        pl.col("away_rolling_pass_epa_5")
    ).alias("rolling_pass_epa_diff_5"),

    # Rolling rushing EPA
    (
        pl.col("home_rolling_rush_epa_5") -
        pl.col("away_rolling_rush_epa_5")
    ).alias("rolling_rush_epa_diff_5"),
])

In [184]:
epa_features = [
    "off_epa_diff",
    "def_epa_diff",
    "rolling_off_epa_diff_5",
    "rolling_def_epa_diff_5",
    "rolling_pass_epa_diff_5",
    "rolling_rush_epa_diff_5",
]

model_data.select([
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    *epa_features,
]).head(10)

game_id,season,week,home_team,away_team,off_epa_diff,def_epa_diff,rolling_off_epa_diff_5,rolling_def_epa_diff_5,rolling_pass_epa_diff_5,rolling_rush_epa_diff_5
str,i32,i32,str,str,f64,f64,f64,f64,f64,f64
"""2015_01_PIT_NE""",2015,1,"""NE""","""PIT""",null,null,null,null,null,null
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""",null,null,null,null,null,null
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""",null,null,null,null,null,null
"""2015_01_KC_HOU""",2015,1,"""HOU""","""KC""",null,null,null,null,null,null
"""2015_01_CAR_JAX""",2015,1,"""JAX""","""CAR""",null,null,null,null,null,null
"""2015_01_CLE_NYJ""",2015,1,"""NYJ""","""CLE""",null,null,null,null,null,null
"""2015_01_SEA_STL""",2015,1,"""STL""","""SEA""",null,null,null,null,null,null
"""2015_01_MIA_WAS""",2015,1,"""WAS""","""MIA""",null,null,null,null,null,null
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""",null,null,null,null,null,null


## V4 - QB 


In [199]:
qb_plays = pbp_regular.filter(
    (pl.col("pass_attempt") == 1) &
    pl.col("passer_player_id").is_not_null()
)

In [200]:
print([
    col for col in pbp_regular.columns
    if "pass" in col.lower() or "passer" in col.lower()
])

['pass_length', 'pass_location', 'total_home_pass_epa', 'total_away_pass_epa', 'total_home_pass_wpa', 'total_away_pass_wpa', 'first_down_pass', 'incomplete_pass', 'pass_attempt', 'pass_touchdown', 'complete_pass', 'passer_player_id', 'passer_player_name', 'passing_yards', 'pass_defense_1_player_id', 'pass_defense_1_player_name', 'pass_defense_2_player_id', 'pass_defense_2_player_name', 'passer', 'passer_jersey_number', 'pass', 'passer_id', 'xpass', 'pass_oe']


In [201]:
qb_game_stats = (
    pbp_regular
    .filter(
        (pl.col("pass_attempt") == 1) &
        pl.col("passer_player_id").is_not_null()
    )
    .group_by([
        "game_id",
        "season",
        "week",
        "posteam",
        "passer_player_id",
        "passer_player_name",
    ])
    .agg([
        pl.col("epa")
        .mean()
        .alias("qb_epa_per_play"),

        pl.col("complete_pass")
        .mean()
        .alias("qb_completion_pct"),

        pl.col("pass_touchdown")
        .mean()
        .alias("qb_td_rate"),

        pl.col("interception")
        .mean()
        .alias("qb_int_rate"),

        pl.len()
        .alias("qb_pass_attempts"),
    ])
    .rename({
        "posteam": "team",
        "passer_player_id": "qb_id",
        "passer_player_name": "qb_name",
    })
)

In [202]:
qb_game_stats.head()

game_id,season,week,team,qb_id,qb_name,qb_epa_per_play,qb_completion_pct,qb_td_rate,qb_int_rate,qb_pass_attempts
str,i32,i32,str,str,str,f64,f64,f64,f64,u32
"""2017_16_TB_CAR""",2017,16,"""TB""","""00-0031503""","""J.Winston""",0.273247,0.636364,0.030303,0.0,33
"""2019_11_JAX_IND""",2019,11,"""JAX""","""00-0029567""","""N.Foles""",-0.113513,0.66,0.04,0.02,50
"""2025_17_LA_ATL""",2025,17,"""LA""","""00-0026498""","""M.Stafford""",-0.067037,0.536585,0.04878,0.073171,41
"""2016_11_BUF_CIN""",2016,11,"""CIN""","""00-0027973""","""A.Dalton""",-0.222041,0.545455,0.022727,0.045455,44
"""2018_14_MIN_SEA""",2018,14,"""SEA""","""00-0029263""","""R.Wilson""",-0.566991,0.434783,0.0,0.043478,23


In [203]:
qb_game_stats.head()

game_id,season,week,team,qb_id,qb_name,qb_epa_per_play,qb_completion_pct,qb_td_rate,qb_int_rate,qb_pass_attempts
str,i32,i32,str,str,str,f64,f64,f64,f64,u32
"""2017_16_TB_CAR""",2017,16,"""TB""","""00-0031503""","""J.Winston""",0.273247,0.636364,0.030303,0.0,33
"""2019_11_JAX_IND""",2019,11,"""JAX""","""00-0029567""","""N.Foles""",-0.113513,0.66,0.04,0.02,50
"""2025_17_LA_ATL""",2025,17,"""LA""","""00-0026498""","""M.Stafford""",-0.067037,0.536585,0.04878,0.073171,41
"""2016_11_BUF_CIN""",2016,11,"""CIN""","""00-0027973""","""A.Dalton""",-0.222041,0.545455,0.022727,0.045455,44
"""2018_14_MIN_SEA""",2018,14,"""SEA""","""00-0029263""","""R.Wilson""",-0.566991,0.434783,0.0,0.043478,23


In [204]:
qb_game_stats = qb_game_stats.filter(
    pl.col("qb_pass_attempts") >= 5
)

qb_game_stats.head()

game_id,season,week,team,qb_id,qb_name,qb_epa_per_play,qb_completion_pct,qb_td_rate,qb_int_rate,qb_pass_attempts
str,i32,i32,str,str,str,f64,f64,f64,f64,u32
"""2017_16_TB_CAR""",2017,16,"""TB""","""00-0031503""","""J.Winston""",0.273247,0.636364,0.030303,0.0,33
"""2019_11_JAX_IND""",2019,11,"""JAX""","""00-0029567""","""N.Foles""",-0.113513,0.66,0.04,0.02,50
"""2025_17_LA_ATL""",2025,17,"""LA""","""00-0026498""","""M.Stafford""",-0.067037,0.536585,0.04878,0.073171,41
"""2016_11_BUF_CIN""",2016,11,"""CIN""","""00-0027973""","""A.Dalton""",-0.222041,0.545455,0.022727,0.045455,44
"""2018_14_MIN_SEA""",2018,14,"""SEA""","""00-0029263""","""R.Wilson""",-0.566991,0.434783,0.0,0.043478,23


In [205]:
qb_game_stats = qb_game_stats.sort([
    "season",
    "qb_id",
    "week"
])

In [206]:
qb_game_stats = (
    pbp_regular
    .filter(
        (pl.col("pass_attempt") == 1) &
        pl.col("passer_player_id").is_not_null()
    )
    .group_by([
        "game_id",
        "season",
        "week",
        "posteam",
        "passer_player_id",
        "passer_player_name",
    ])
    .agg([
        pl.col("epa").sum().alias("qb_epa"),
        pl.len().alias("qb_pass_attempts"),
        pl.col("complete_pass").sum().alias("qb_completions"),
        pl.col("pass_touchdown").sum().alias("qb_pass_tds"),
        pl.col("interception").sum().alias("qb_interceptions"),
    ])
    .rename({
        "posteam": "team",
        "passer_player_id": "qb_id",
        "passer_player_name": "qb_name",
    })
    .sort(["season", "qb_id", "week"])
)

In [211]:
qb_game_stats = qb_game_stats.with_columns([
    pl.col("qb_epa")
    .cum_sum()
    .over(["season", "qb_id"])
    .alias("qb_epa_cumulative"),

    pl.col("qb_pass_attempts")
    .cum_sum()
    .over(["season", "qb_id"])
    .alias("qb_attempts_cumulative"),

    pl.col("qb_completions")
    .cum_sum()
    .over(["season", "qb_id"])
    .alias("qb_completions_cumulative"),

    pl.col("qb_pass_tds")
    .cum_sum()
    .over(["season", "qb_id"])
    .alias("qb_tds_cumulative"),

    pl.col("qb_interceptions")
    .cum_sum()
    .over(["season", "qb_id"])
    .alias("qb_interceptions_cumulative"),
])

In [212]:
qb_game_stats = qb_game_stats.with_columns([
    pl.col("qb_epa_cumulative")
    .shift(1)
    .over(["season", "qb_id"])
    .alias("qb_epa_before"),

    pl.col("qb_attempts_cumulative")
    .shift(1)
    .over(["season", "qb_id"])
    .alias("qb_attempts_before"),

    pl.col("qb_completions_cumulative")
    .shift(1)
    .over(["season", "qb_id"])
    .alias("qb_completions_before"),

    pl.col("qb_tds_cumulative")
    .shift(1)
    .over(["season", "qb_id"])
    .alias("qb_tds_before"),

    pl.col("qb_interceptions_cumulative")
    .shift(1)
    .over(["season", "qb_id"])
    .alias("qb_interceptions_before"),
])

In [213]:
qb_game_stats = qb_game_stats.with_columns([
    (
        pl.col("qb_epa_before") /
        pl.col("qb_attempts_before")
    ).alias("qb_epa_before"),

    (
        pl.col("qb_completions_before") /
        pl.col("qb_attempts_before")
    ).alias("qb_completion_before"),

    (
        pl.col("qb_tds_before") /
        pl.col("qb_attempts_before")
    ).alias("qb_td_rate_before"),

    (
        pl.col("qb_interceptions_before") /
        pl.col("qb_attempts_before")
    ).alias("qb_int_rate_before"),
])

In [214]:
qb_game_stats.select([
    "season",
    "week",
    "qb_name",
    "qb_pass_attempts",
    "qb_epa_before",
    "qb_completion_before",
    "qb_td_rate_before",
    "qb_int_rate_before",
    "qb_rolling_epa_5",
    "qb_rolling_completion_5",
    "qb_rolling_td_rate_5",
    "qb_rolling_int_rate_5",
]).head(10)

season,week,qb_name,qb_pass_attempts,qb_epa_before,qb_completion_before,qb_td_rate_before,qb_int_rate_before,qb_rolling_epa_5,qb_rolling_completion_5,qb_rolling_td_rate_5,qb_rolling_int_rate_5
i32,i32,str,u32,f64,f64,f64,f64,f64,f64,f64,f64
2015,4,"""M.Hasselbeck""",50,null,null,null,null,null,null,null,null
2015,5,"""M.Hasselbeck""",29,0.068393,0.6,0.02,0.0,0.068393,0.6,0.02,0.0
2015,11,"""M.Hasselbeck""",34,0.180616,0.607595,0.037975,0.0,0.180616,0.607595,0.037975,0.0
2015,12,"""M.Hasselbeck""",45,0.121077,0.628319,0.044248,0.017699,0.121077,0.628319,0.044248,0.017699
2015,13,"""M.Hasselbeck""",28,0.150218,0.613924,0.044304,0.012658,0.150218,0.613924,0.044304,0.012658
2015,14,"""M.Hasselbeck""",38,0.069482,0.607527,0.043011,0.021505,0.069482,0.607527,0.043011,0.021505
2015,15,"""M.Hasselbeck""",32,0.005414,0.584821,0.035714,0.017857,-0.012683,0.58046,0.04023,0.022989
2015,16,"""M.Hasselbeck""",16,-0.030725,0.578125,0.035156,0.019531,-0.125053,0.564972,0.033898,0.028249
2015,1,"""P.Manning""",44,null,null,null,null,null,null,null,null


In [215]:
qb_features = qb_game_stats.select([
    "game_id",
    "qb_id",
    "qb_epa_before",
    "qb_completion_before",
    "qb_td_rate_before",
    "qb_int_rate_before",
    "qb_rolling_epa_5",
    "qb_rolling_completion_5",
    "qb_rolling_td_rate_5",
    "qb_rolling_int_rate_5",
])

In [216]:
home_qb_features = qb_features.rename({
    "qb_id": "home_qb_id",
    "qb_epa_before": "home_qb_epa",
    "qb_completion_before": "home_qb_completion",
    "qb_td_rate_before": "home_qb_td_rate",
    "qb_int_rate_before": "home_qb_int_rate",
    "qb_rolling_epa_5": "home_qb_rolling_epa_5",
    "qb_rolling_completion_5": "home_qb_rolling_completion_5",
    "qb_rolling_td_rate_5": "home_qb_rolling_td_rate_5",
    "qb_rolling_int_rate_5": "home_qb_rolling_int_rate_5",
})

away_qb_features = qb_features.rename({
    "qb_id": "away_qb_id",
    "qb_epa_before": "away_qb_epa",
    "qb_completion_before": "away_qb_completion",
    "qb_td_rate_before": "away_qb_td_rate",
    "qb_int_rate_before": "away_qb_int_rate",
    "qb_rolling_epa_5": "away_qb_rolling_epa_5",
    "qb_rolling_completion_5": "away_qb_rolling_completion_5",
    "qb_rolling_td_rate_5": "away_qb_rolling_td_rate_5",
    "qb_rolling_int_rate_5": "away_qb_rolling_int_rate_5",
})

In [218]:
home_qb_features.select([
    "game_id",
    "home_qb_id",
    "home_qb_epa",
    "home_qb_completion",
]).head()

game_id,home_qb_id,home_qb_epa,home_qb_completion
str,str,f64,f64
"""2015_04_JAX_IND""","""00-0007091""",null,null
"""2015_05_IND_HOU""","""00-0007091""",0.068393,0.6
"""2015_11_IND_ATL""","""00-0007091""",0.180616,0.607595
"""2015_12_TB_IND""","""00-0007091""",0.121077,0.628319
"""2015_13_IND_PIT""","""00-0007091""",0.150218,0.613924


In [220]:
model_data = model_data.join(
    home_qb_features,
    on=["game_id", "home_qb_id"],
    how="left"
)

model_data = model_data.join(
    away_qb_features,
    on=["game_id", "away_qb_id"],
    how="left"
)

In [221]:
print([
    col for col in model_data.columns
    if "qb_" in col
])

['away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'home_qb_epa', 'home_qb_completion', 'home_qb_td_rate', 'home_qb_int_rate', 'home_qb_rolling_epa_5', 'home_qb_rolling_completion_5', 'home_qb_rolling_td_rate_5', 'home_qb_rolling_int_rate_5', 'away_qb_epa', 'away_qb_completion', 'away_qb_td_rate', 'away_qb_int_rate', 'away_qb_rolling_epa_5', 'away_qb_rolling_completion_5', 'away_qb_rolling_td_rate_5', 'away_qb_rolling_int_rate_5']


In [222]:
model_data.select([
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    "home_qb_name",
    "away_qb_name",
    "home_qb_epa",
    "away_qb_epa",
    "home_qb_completion",
    "away_qb_completion",
]).head(15)

game_id,season,week,home_team,away_team,home_qb_name,away_qb_name,home_qb_epa,away_qb_epa,home_qb_completion,away_qb_completion
str,i32,i32,str,str,str,str,f64,f64,f64,f64
"""2015_01_PIT_NE""",2015,1,"""NE""","""PIT""","""Tom Brady""","""Ben Roethlisberger""",null,null,null,null
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""","""Tyrod Taylor""","""Andrew Luck""",null,null,null,null
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""","""Jay Cutler""","""Aaron Rodgers""",null,null,null,null
"""2015_01_KC_HOU""",2015,1,"""HOU""","""KC""","""Brian Hoyer""","""Alex Smith""",null,null,null,null
"""2015_01_CAR_JAX""",2015,1,"""JAX""","""CAR""","""Blake Bortles""","""Cam Newton""",null,null,null,null
…,…,…,…,…,…,…,…,…,…,…
"""2015_01_BAL_DEN""",2015,1,"""DEN""","""BAL""","""Peyton Manning""","""Joe Flacco""",null,null,null,null
"""2015_01_CIN_OAK""",2015,1,"""OAK""","""CIN""","""Derek Carr""","""Andy Dalton""",null,null,null,null
"""2015_01_TEN_TB""",2015,1,"""TB""","""TEN""","""Jameis Winston""","""Marcus Mariota""",null,null,null,null


In [223]:
model_data = model_data.with_columns([
    # Season-to-date QB performance
    (
        pl.col("home_qb_epa") -
        pl.col("away_qb_epa")
    ).alias("qb_epa_diff"),

    (
        pl.col("home_qb_completion") -
        pl.col("away_qb_completion")
    ).alias("qb_completion_diff"),

    (
        pl.col("home_qb_td_rate") -
        pl.col("away_qb_td_rate")
    ).alias("qb_td_rate_diff"),

    # Lower INT rate is better, so reverse the difference
    (
        pl.col("away_qb_int_rate") -
        pl.col("home_qb_int_rate")
    ).alias("qb_int_rate_diff"),

    # Recent QB form
    (
        pl.col("home_qb_rolling_epa_5") -
        pl.col("away_qb_rolling_epa_5")
    ).alias("qb_rolling_epa_diff_5"),

    (
        pl.col("home_qb_rolling_completion_5") -
        pl.col("away_qb_rolling_completion_5")
    ).alias("qb_rolling_completion_diff_5"),

    (
        pl.col("home_qb_rolling_td_rate_5") -
        pl.col("away_qb_rolling_td_rate_5")
    ).alias("qb_rolling_td_rate_diff_5"),

    (
        pl.col("away_qb_rolling_int_rate_5") -
        pl.col("home_qb_rolling_int_rate_5")
    ).alias("qb_rolling_int_rate_diff_5"),
])

In [225]:
baseline_features = [
    "win_pct_diff",
    "ppg_diff",
    "defense_diff",
]

rolling_features = [
    "rolling_win_pct_diff_5",
    "rolling_ppg_diff_5",
    "rolling_defense_diff_5",
    "rolling_point_diff_diff_5",
]

epa_features = [
    "off_epa_diff",
    "def_epa_diff",
    "rolling_off_epa_diff_5",
    "rolling_def_epa_diff_5",
    "rolling_pass_epa_diff_5",
    "rolling_rush_epa_diff_5",
]

In [226]:
v4_features = (
    baseline_features
    + rolling_features
    + epa_features
    + [
        "qb_epa_diff",
        "qb_completion_diff",
        "qb_td_rate_diff",
        "qb_int_rate_diff",
        "qb_rolling_epa_diff_5",
        "qb_rolling_completion_diff_5",
        "qb_rolling_td_rate_diff_5",
        "qb_rolling_int_rate_diff_5",
    ]
)

In [227]:
v4_features = (
    baseline_features
    + rolling_features
    + epa_features
    + [
        "qb_epa_diff",
        "qb_completion_diff",
        "qb_td_rate_diff",
        "qb_int_rate_diff",
        "qb_rolling_epa_diff_5",
        "qb_rolling_completion_diff_5",
        "qb_rolling_td_rate_diff_5",
        "qb_rolling_int_rate_diff_5",
    ]
)

print(f"Number of V4 features: {len(v4_features)}")
print(v4_features)

Number of V4 features: 21
['win_pct_diff', 'ppg_diff', 'defense_diff', 'rolling_win_pct_diff_5', 'rolling_ppg_diff_5', 'rolling_defense_diff_5', 'rolling_point_diff_diff_5', 'off_epa_diff', 'def_epa_diff', 'rolling_off_epa_diff_5', 'rolling_def_epa_diff_5', 'rolling_pass_epa_diff_5', 'rolling_rush_epa_diff_5', 'qb_epa_diff', 'qb_completion_diff', 'qb_td_rate_diff', 'qb_int_rate_diff', 'qb_rolling_epa_diff_5', 'qb_rolling_completion_diff_5', 'qb_rolling_td_rate_diff_5', 'qb_rolling_int_rate_diff_5']


## SAVING

In [229]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "model_data.parquet"

model_data.write_parquet(output_path)

print(f"Saved to: {output_path.resolve()}")
print(f"File exists: {output_path.exists()}")

Saved to: /Users/joaquin/NFL-GamePredictor/data/processed/model_data.parquet
File exists: True
